In [1]:
import pandas as pd
import os

def load_physionet_data(path):
    all_data = []
    for file in os.listdir(path):
        if file.endswith(".psv"):
            patient_id = file.replace(".psv", "")
            df = pd.read_csv(os.path.join(path, file), sep='|')
            df["Patient_ID"] = patient_id
            all_data.append(df)
    return pd.concat(all_data, ignore_index=True)

train_A_path = r"C:\Users\gulsh\OneDrive\Desktop\research projects\sepsis prediction\training\training_setA"
train_B_path = r"C:\Users\gulsh\OneDrive\Desktop\research projects\sepsis prediction\training\training_setB"

df_A = load_physionet_data(train_A_path)
df_B = load_physionet_data(train_B_path)

df = pd.concat([df_A, df_B], ignore_index=True)
print(df.shape)
df.head()

(1552210, 42)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001


LSTM

In [2]:
import numpy as np
import pandas as pd
import gc, warnings, time, os, psutil
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# =========================
# GLOBAL SETTINGS
# =========================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
WINDOW = 12 
STRIDE = 6
POS_WEIGHT = 2.0
ALPHA = 0.2
EPOCHS = 120
BATCH_SIZE = 8

TARGET_AUROC = 0.85  # for convergence time 

# LSTM Settings
HIDDEN_DIM = 32
NUM_LAYERS = 1
BIDIRECTIONAL = True

# =========================
# STEP 1: CLINICAL LABELING
# =========================
def apply_clinical_labeling(df):
    df = df.copy()
    df["SepsisLabel_shifted"] = 0
    df["use_for_training"] = 1

    for pid, g in df.groupby("Patient_ID"):
        g = g.reset_index()
        onset_idx = g[g["SepsisLabel"] == 1].index

        for t in onset_idx:
            pos_start = max(0, t - 3)
            df.loc[g.loc[pos_start:t, "index"], "SepsisLabel_shifted"] = 1

            amb_start = max(0, t - 6)
            df.loc[g.loc[amb_start:pos_start-1, "index"], "use_for_training"] = 0

    return df

# =========================
# STEP 2: PREPROCESSING
# =========================
FEATURES = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
    'pH','PaCO2','BUN','Creatinine','Lactate',
    'WBC','Platelets','Age'
]

def preprocess(df):
    df = df.copy()

    for c in FEATURES:
        df[c] = df.groupby("Patient_ID")[c].ffill().bfill()
        df[c] = df[c].fillna(df.groupby("Patient_ID")[c].transform("mean"))

    df["ShockIndex"] = df["HR"] / df["SBP"].replace(0, 1)
    df["ModShockIndex"] = df["HR"] / df["MAP"].replace(0, 1)
    df["PulsePressure"] = df["SBP"] - df["DBP"]

    new_features = ["HR", "MAP", "Resp", "ShockIndex", "ModShockIndex"]

    for c in new_features:
        df[f"{c}_diff6"] = df.groupby("Patient_ID")[c].diff(6).fillna(0)

    return df

ALL_FEATURES = FEATURES + [
    "HR_diff6", "MAP_diff6", "Resp_diff6",
    "ShockIndex", "ModShockIndex", "ShockIndex_diff6", "ModShockIndex_diff6"
]

# =========================
# STEP 3: WINDOWING
# =========================
def make_windows(df, balance=False, max_per_patient=16, min_signal=0.15):
    X_pos, X_neg = [], []

    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < WINDOW:
            continue

        n = min((len(g)-WINDOW)//STRIDE + 1, max_per_patient)
        collected = 0

        for i in range(0, len(g)-WINDOW+1, STRIDE):
            if collected >= n:
                break

            x = g.iloc[i:i+WINDOW][ALL_FEATURES].values.astype(np.float32)
            y = g.iloc[i:i+WINDOW]["SepsisLabel_shifted"].max()

            if balance:
                signal = np.abs(x[-1] - x[-7]).mean()
                if signal < min_signal:
                    continue

            (X_pos if y == 1 else X_neg).append(x)
            collected += 1

    if balance:
        n = min(len(X_pos), len(X_neg))
        X_pos, X_neg = X_pos[:n], X_neg[:n]

    X = np.array(X_pos + X_neg)
    y = np.array([1]*len(X_pos) + [0]*len(X_neg))
    return X, y

# =========================
# STEP 4: CLASSICAL LSTM MODEL
# =========================
class ClassicalLSTM(nn.Module):
    def __init__(self, n_features, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, bidirectional=BIDIRECTIONAL):
        super().__init__()
        
        # LSTM processes the entire time window sequentially
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        
        # Multiply hidden_dim by 2 if bidirectional
        lstm_out_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.cls = nn.Linear(lstm_out_dim, 1)

    def forward(self, x):
        # x shape: (batch_size, window_size, features)
        out, (h_n, c_n) = self.lstm(x)
        
        # Extract the output of the final time step in the window
        last_out = out[:, -1, :]
        
        logits = self.cls(last_out)
        return logits.squeeze(1)

# =========================
# LOSS
# =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.gamma = gamma

    def forward(self, logits, y):
        bce = self.bce(logits, y)
        pt = torch.exp(-bce)
        return ((1-pt)**self.gamma * bce).mean()

# =========================
# 5-FOLD PATIENT CV + ENSEMBLE
# =========================
from sklearn.model_selection import GroupShuffleSplit
import copy

process = psutil.Process(os.getpid())

# 1. Prepare Data
df = preprocess(apply_clinical_labeling(df))

# 2. Create Hold-Out Test Set (20%)
print("✂️ Splitting Hold-Out Test Set (20%)...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df.Patient_ID))

df_dev = df.iloc[train_idx]
df_test = df.iloc[test_idx]

print(f"Dataset Shape: Dev {df_dev.shape}, Test {df_test.shape}")

gkf = GroupKFold(n_splits=5)
saved_checkpoints = []

# --- TRAINING TIME / CONVERGENCE METRICS ---
global_train_start = time.time()
converged_fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_dev, groups=df_dev.Patient_ID)):
    print(f"\n🚀 Training Ensemble Model {fold+1}/5...")

    df_tr = df_dev.iloc[tr_idx]

    Xtr, ytr = make_windows(df_tr, balance=True, max_per_patient=16)

    scaler = StandardScaler()
    # Scale across (Batch * Window, Features) to preserve temporal normalization
    Xtr = scaler.fit_transform(Xtr.reshape(-1, Xtr.shape[-1])).reshape(Xtr.shape)

    model = ClassicalLSTM(Xtr.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))
    focal = FocalLoss()

    def loss_fn(logits, y):
        return ALPHA*bce(logits, y) + (1-ALPHA)*focal(logits, y)

    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).float()),
        batch_size=BATCH_SIZE, shuffle=True
    )

    converged_epoch = None
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        model.train()
        all_logits = []
        all_targets = []

        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            all_logits.append(logits.detach().cpu())
            all_targets.append(yb.detach().cpu())

        epoch_time = time.time() - epoch_start
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Fold {fold+1}, Epoch {epoch+1}/{EPOCHS}, epoch time: {epoch_time:.3f} s, loss: {loss.item():.4f}")

        with torch.no_grad():
            probs_epoch = torch.sigmoid(torch.cat(all_logits)).numpy()
            y_epoch = torch.cat(all_targets).numpy()
            try:
                train_auroc_epoch = roc_auc_score(y_epoch, probs_epoch)
            except ValueError:
                train_auroc_epoch = np.nan

        if converged_epoch is None and not np.isnan(train_auroc_epoch) and train_auroc_epoch >= TARGET_AUROC:
            converged_epoch = epoch + 1  
            
    converged_fold_epochs.append(converged_epoch)

    checkpoint = {
        'model_state': copy.deepcopy(model.state_dict()),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }
    saved_checkpoints.append(checkpoint)
    print(f"✅ Model {fold+1} Saved. Converged epoch (train AUROC ≥ {TARGET_AUROC}): {converged_epoch}")

global_train_end = time.time()
total_training_time = global_train_end - global_train_start
print(f"\n⏱ Total training time (5 folds): {total_training_time/60:.2f} minutes")

# =========================
# ENSEMBLE VOTING (INFERENCE)
# =========================
print(f"\n🔮 Running Ensemble Voting on {df_test.Patient_ID.nunique()} Test Patients...")

X_test_raw, y_test = make_windows(df_test, balance=False, max_per_patient=999)

ensemble_probs = np.zeros(len(y_test))

# --- Inference time & resource tracking ---
mem_before = process.memory_info().rss
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

inf_start = time.time()

for i, checkpoint in enumerate(saved_checkpoints):
    scaler = StandardScaler()
    scaler.mean_ = checkpoint['scaler_mean']
    scaler.scale_ = checkpoint['scaler_scale']

    X_test_scaled = scaler.transform(
        X_test_raw.reshape(-1, X_test_raw.shape[-1])
    ).reshape(X_test_raw.shape)

    model = ClassicalLSTM(X_test_scaled.shape[-1]).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    with torch.no_grad():
        logits = model(torch.tensor(X_test_scaled).to(DEVICE))
        logits = torch.nan_to_num(logits, nan=0.0, posinf=10.0, neginf=-10.0)
        probs = torch.sigmoid(logits).cpu().numpy()

    ensemble_probs += probs

if torch.cuda.is_available():
    torch.cuda.synchronize()
inf_end = time.time()

ensemble_probs /= len(saved_checkpoints)

mem_after = process.memory_info().rss
cpu_mem_used = (mem_after - mem_before) / (1024**2)  # MB

if torch.cuda.is_available():
    gpu_peak_mem = torch.cuda.max_memory_allocated() / (1024**2)
else:
    gpu_peak_mem = None

inference_time = inf_end - inf_start
latency_per_sample = inference_time / len(y_test)

# Model size (parameters only, single model)
model_for_size = ClassicalLSTM(X_test_raw.shape[-1]).to(DEVICE)
total_params = sum(p.numel() for p in model_for_size.parameters())
model_size_mb = total_params * 4 / (1024**2)

print("\n⏱ Time Metrics")
print(f"Inference time (ensemble, test set): {inference_time:.4f} s")
print(f"Latency per sample: {latency_per_sample*1e3:.3f} ms")
print(f"Total training time (all folds): {total_training_time:.2f} s")
print(f"Converged epochs per fold: {converged_fold_epochs}")

print("\n💾 Resource Metrics")
print(f"CPU memory used during inference: {cpu_mem_used:.2f} MB")
if gpu_peak_mem is not None:
    print(f"GPU peak memory during inference: {gpu_peak_mem:.2f} MB")
print(f"Model size (single model parameters): {model_size_mb:.2f} MB")
print(f"Total parameters: {total_params}")

# =========================
# FINAL METRICS
# =========================
auroc = roc_auc_score(y_test, ensemble_probs)
auprc = average_precision_score(y_test, ensemble_probs)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_probs > 0.5, average="binary", zero_division=0
)

print("\n" + "="*40)
print(f"🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟")
print("="*40)
print(f"Test AUROC: {auroc:.3f}")
print(f"Test AUPRC: {auprc:.3f}")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
print("="*40)

✂️ Splitting Hold-Out Test Set (20%)...
Dataset Shape: Dev (1241213, 52), Test (310997, 52)

🚀 Training Ensemble Model 1/5...
Fold 1, Epoch 1/120, epoch time: 3.396 s, loss: 0.2796
Fold 1, Epoch 10/120, epoch time: 1.148 s, loss: 0.1223
Fold 1, Epoch 20/120, epoch time: 1.025 s, loss: 0.0268
Fold 1, Epoch 30/120, epoch time: 1.049 s, loss: 0.0109
Fold 1, Epoch 40/120, epoch time: 1.045 s, loss: 0.0182
Fold 1, Epoch 50/120, epoch time: 1.048 s, loss: 0.0010
Fold 1, Epoch 60/120, epoch time: 1.061 s, loss: 0.0004
Fold 1, Epoch 70/120, epoch time: 1.126 s, loss: 0.0003
Fold 1, Epoch 80/120, epoch time: 1.114 s, loss: 0.0001
Fold 1, Epoch 90/120, epoch time: 1.036 s, loss: 0.0000
Fold 1, Epoch 100/120, epoch time: 1.042 s, loss: 0.0133
Fold 1, Epoch 110/120, epoch time: 1.061 s, loss: 0.0001
Fold 1, Epoch 120/120, epoch time: 1.152 s, loss: 0.0005
✅ Model 1 Saved. Converged epoch (train AUROC ≥ 0.85): 4

🚀 Training Ensemble Model 2/5...
Fold 2, Epoch 1/120, epoch time: 1.163 s, loss: 0.302

CNN

In [3]:
import numpy as np
import pandas as pd
import gc, warnings, time, os, psutil
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# =========================
# GLOBAL SETTINGS
# =========================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
WINDOW = 12 
STRIDE = 6
POS_WEIGHT = 2.0
ALPHA = 0.2
EPOCHS = 120
BATCH_SIZE = 8

TARGET_AUROC = 0.85  # for convergence time tracking

# =========================
# STEP 1: CLINICAL LABELING
# =========================
def apply_clinical_labeling(df):
    df = df.copy()
    df["SepsisLabel_shifted"] = 0
    df["use_for_training"] = 1

    for pid, g in df.groupby("Patient_ID"):
        g = g.reset_index()
        onset_idx = g[g["SepsisLabel"] == 1].index

        for t in onset_idx:
            pos_start = max(0, t - 3)
            df.loc[g.loc[pos_start:t, "index"], "SepsisLabel_shifted"] = 1

            amb_start = max(0, t - 6)
            df.loc[g.loc[amb_start:pos_start-1, "index"], "use_for_training"] = 0

    return df

# =========================
# STEP 2: PREPROCESSING
# =========================
FEATURES = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
    'pH','PaCO2','BUN','Creatinine','Lactate',
    'WBC','Platelets','Age'
]

def preprocess(df):
    df = df.copy()

    for c in FEATURES:
        df[c] = df.groupby("Patient_ID")[c].ffill().bfill()
        df[c] = df[c].fillna(df.groupby("Patient_ID")[c].transform("mean"))

    df["ShockIndex"] = df["HR"] / df["SBP"].replace(0, 1)
    df["ModShockIndex"] = df["HR"] / df["MAP"].replace(0, 1)
    df["PulsePressure"] = df["SBP"] - df["DBP"]

    new_features = ["HR", "MAP", "Resp", "ShockIndex", "ModShockIndex"]

    for c in new_features:
        df[f"{c}_diff6"] = df.groupby("Patient_ID")[c].diff(6).fillna(0)

    return df

ALL_FEATURES = FEATURES + [
    "HR_diff6", "MAP_diff6", "Resp_diff6",
    "ShockIndex", "ModShockIndex", "ShockIndex_diff6", "ModShockIndex_diff6"
]

# =========================
# STEP 3: WINDOWING
# =========================
def make_windows(df, balance=False, max_per_patient=16, min_signal=0.15):
    X_pos, X_neg = [], []

    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < WINDOW:
            continue

        n = min((len(g)-WINDOW)//STRIDE + 1, max_per_patient)
        collected = 0

        for i in range(0, len(g)-WINDOW+1, STRIDE):
            if collected >= n:
                break

            x = g.iloc[i:i+WINDOW][ALL_FEATURES].values.astype(np.float32)
            y = g.iloc[i:i+WINDOW]["SepsisLabel_shifted"].max()

            if balance:
                signal = np.abs(x[-1] - x[-7]).mean()
                if signal < min_signal:
                    continue

            (X_pos if y == 1 else X_neg).append(x)
            collected += 1

    if balance:
        n = min(len(X_pos), len(X_neg))
        X_pos, X_neg = X_pos[:n], X_neg[:n]

    X = np.array(X_pos + X_neg)
    y = np.array([1]*len(X_pos) + [0]*len(X_neg))
    return X, y

# =========================
# STEP 4: CLASSICAL 1D CNN MODEL
# =========================
class ClassicalCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        
        # Conv1d expects input shape: (Batch_Size, Channels, Length)
        # Channels = Number of Features, Length = Window Size
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        # Adaptive pooling squashes whatever time length is left into exactly 1
        self.pool2 = nn.AdaptiveAvgPool1d(1) 
        
        self.cls = nn.Linear(128, 1)

    def forward(self, x):
        # x is currently (Batch, Window, Features)
        # We must permute to (Batch, Features, Window) for Conv1d
        x = x.permute(0, 2, 1)
        
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x) # Output shape: (Batch, 128, 1)
        
        x = x.squeeze(-1) # Flatten to (Batch, 128)
        
        logits = self.cls(x)
        return logits.squeeze(1)

# =========================
# LOSS
# =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.gamma = gamma

    def forward(self, logits, y):
        bce = self.bce(logits, y)
        pt = torch.exp(-bce)
        return ((1-pt)**self.gamma * bce).mean()

# =========================
# 5-FOLD PATIENT CV + ENSEMBLE
# =========================
from sklearn.model_selection import GroupShuffleSplit
import copy

process = psutil.Process(os.getpid())

# 1. Prepare Data
df = preprocess(apply_clinical_labeling(df))

# 2. Create Hold-Out Test Set (20%)
print("✂️ Splitting Hold-Out Test Set (20%)...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df.Patient_ID))

df_dev = df.iloc[train_idx]
df_test = df.iloc[test_idx]

print(f"Dataset Shape: Dev {df_dev.shape}, Test {df_test.shape}")

gkf = GroupKFold(n_splits=5)
saved_checkpoints = []

# --- TRAINING TIME / CONVERGENCE METRICS ---
global_train_start = time.time()
converged_fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_dev, groups=df_dev.Patient_ID)):
    print(f"\n🚀 Training Ensemble Model {fold+1}/5...")

    df_tr = df_dev.iloc[tr_idx]

    Xtr, ytr = make_windows(df_tr, balance=True, max_per_patient=16)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr.reshape(-1, Xtr.shape[-1])).reshape(Xtr.shape)

    model = ClassicalCNN(Xtr.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))
    focal = FocalLoss()

    def loss_fn(logits, y):
        return ALPHA*bce(logits, y) + (1-ALPHA)*focal(logits, y)

    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).float()),
        batch_size=BATCH_SIZE, shuffle=True
    )

    converged_epoch = None
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        model.train()
        all_logits = []
        all_targets = []

        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            all_logits.append(logits.detach().cpu())
            all_targets.append(yb.detach().cpu())

        epoch_time = time.time() - epoch_start
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Fold {fold+1}, Epoch {epoch+1}/{EPOCHS}, epoch time: {epoch_time:.3f} s, loss: {loss.item():.4f}")

        with torch.no_grad():
            probs_epoch = torch.sigmoid(torch.cat(all_logits)).numpy()
            y_epoch = torch.cat(all_targets).numpy()
            try:
                train_auroc_epoch = roc_auc_score(y_epoch, probs_epoch)
            except ValueError:
                train_auroc_epoch = np.nan

        if converged_epoch is None and not np.isnan(train_auroc_epoch) and train_auroc_epoch >= TARGET_AUROC:
            converged_epoch = epoch + 1  
            
    converged_fold_epochs.append(converged_epoch)

    checkpoint = {
        'model_state': copy.deepcopy(model.state_dict()),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }
    saved_checkpoints.append(checkpoint)
    print(f"✅ Model {fold+1} Saved. Converged epoch (train AUROC ≥ {TARGET_AUROC}): {converged_epoch}")

global_train_end = time.time()
total_training_time = global_train_end - global_train_start
print(f"\n⏱ Total training time (5 folds): {total_training_time/60:.2f} minutes")

# =========================
# ENSEMBLE VOTING (INFERENCE)
# =========================
print(f"\n🔮 Running Ensemble Voting on {df_test.Patient_ID.nunique()} Test Patients...")

X_test_raw, y_test = make_windows(df_test, balance=False, max_per_patient=999)

ensemble_probs = np.zeros(len(y_test))

# --- Inference time & resource tracking ---
mem_before = process.memory_info().rss
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

inf_start = time.time()

for i, checkpoint in enumerate(saved_checkpoints):
    scaler = StandardScaler()
    scaler.mean_ = checkpoint['scaler_mean']
    scaler.scale_ = checkpoint['scaler_scale']

    X_test_scaled = scaler.transform(
        X_test_raw.reshape(-1, X_test_raw.shape[-1])
    ).reshape(X_test_raw.shape)

    model = ClassicalCNN(X_test_scaled.shape[-1]).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    with torch.no_grad():
        logits = model(torch.tensor(X_test_scaled).to(DEVICE))
        logits = torch.nan_to_num(logits, nan=0.0, posinf=10.0, neginf=-10.0)
        probs = torch.sigmoid(logits).cpu().numpy()

    ensemble_probs += probs

if torch.cuda.is_available():
    torch.cuda.synchronize()
inf_end = time.time()

ensemble_probs /= len(saved_checkpoints)

mem_after = process.memory_info().rss
cpu_mem_used = (mem_after - mem_before) / (1024**2)  # MB

if torch.cuda.is_available():
    gpu_peak_mem = torch.cuda.max_memory_allocated() / (1024**2)
else:
    gpu_peak_mem = None

inference_time = inf_end - inf_start
latency_per_sample = inference_time / len(y_test)

# Model size (parameters only, single model)
model_for_size = ClassicalCNN(X_test_raw.shape[-1]).to(DEVICE)
total_params = sum(p.numel() for p in model_for_size.parameters())
model_size_mb = total_params * 4 / (1024**2)

print("\n⏱ Time Metrics")
print(f"Inference time (ensemble, test set): {inference_time:.4f} s")
print(f"Latency per sample: {latency_per_sample*1e3:.3f} ms")
print(f"Total training time (all folds): {total_training_time:.2f} s")
print(f"Converged epochs per fold: {converged_fold_epochs}")

print("\n💾 Resource Metrics")
print(f"CPU memory used during inference: {cpu_mem_used:.2f} MB")
if gpu_peak_mem is not None:
    print(f"GPU peak memory during inference: {gpu_peak_mem:.2f} MB")
print(f"Model size (single model parameters): {model_size_mb:.2f} MB")
print(f"Total parameters: {total_params}")

# =========================
# FINAL METRICS
# =========================
auroc = roc_auc_score(y_test, ensemble_probs)
auprc = average_precision_score(y_test, ensemble_probs)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_probs > 0.5, average="binary", zero_division=0
)

print("\n" + "="*40)
print(f"🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟")
print("="*40)
print(f"Test AUROC: {auroc:.3f}")
print(f"Test AUPRC: {auprc:.3f}")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
print("="*40)

✂️ Splitting Hold-Out Test Set (20%)...
Dataset Shape: Dev (1241213, 52), Test (310997, 52)

🚀 Training Ensemble Model 1/5...
Fold 1, Epoch 1/120, epoch time: 0.686 s, loss: 0.3403
Fold 1, Epoch 10/120, epoch time: 0.516 s, loss: 0.0763
Fold 1, Epoch 20/120, epoch time: 0.588 s, loss: 0.0149
Fold 1, Epoch 30/120, epoch time: 0.513 s, loss: 0.0187
Fold 1, Epoch 40/120, epoch time: 0.506 s, loss: 0.0064
Fold 1, Epoch 50/120, epoch time: 0.516 s, loss: 0.0006
Fold 1, Epoch 60/120, epoch time: 0.523 s, loss: 0.0000
Fold 1, Epoch 70/120, epoch time: 0.514 s, loss: 0.0001
Fold 1, Epoch 80/120, epoch time: 0.516 s, loss: 0.0000
Fold 1, Epoch 90/120, epoch time: 0.513 s, loss: 0.0000
Fold 1, Epoch 100/120, epoch time: 0.532 s, loss: 0.0000
Fold 1, Epoch 110/120, epoch time: 0.597 s, loss: 0.0000
Fold 1, Epoch 120/120, epoch time: 0.527 s, loss: 0.0000
✅ Model 1 Saved. Converged epoch (train AUROC ≥ 0.85): 4

🚀 Training Ensemble Model 2/5...
Fold 2, Epoch 1/120, epoch time: 0.544 s, loss: 0.348

TRANSFORMER

In [4]:
import numpy as np
import pandas as pd
import gc, warnings, time, os, psutil
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# =========================
# GLOBAL SETTINGS
# =========================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
WINDOW = 12 
STRIDE = 6
POS_WEIGHT = 2.0
ALPHA = 0.2
EPOCHS = 120
BATCH_SIZE = 8

TARGET_AUROC = 0.85  # for convergence time tracking

# Transformer Settings
D_MODEL = 64         # Embedding dimension
N_HEADS = 4          # Number of attention heads
NUM_LAYERS = 2       # Number of transformer encoder blocks
DIM_FEEDFORWARD = 128 # Hidden layer size inside the transformer block
DROPOUT = 0.1

# =========================
# STEP 1: CLINICAL LABELING
# =========================
def apply_clinical_labeling(df):
    df = df.copy()
    df["SepsisLabel_shifted"] = 0
    df["use_for_training"] = 1

    for pid, g in df.groupby("Patient_ID"):
        g = g.reset_index()
        onset_idx = g[g["SepsisLabel"] == 1].index

        for t in onset_idx:
            pos_start = max(0, t - 3)
            df.loc[g.loc[pos_start:t, "index"], "SepsisLabel_shifted"] = 1

            amb_start = max(0, t - 6)
            df.loc[g.loc[amb_start:pos_start-1, "index"], "use_for_training"] = 0

    return df

# =========================
# STEP 2: PREPROCESSING
# =========================
FEATURES = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
    'pH','PaCO2','BUN','Creatinine','Lactate',
    'WBC','Platelets','Age'
]

def preprocess(df):
    df = df.copy()

    for c in FEATURES:
        df[c] = df.groupby("Patient_ID")[c].ffill().bfill()
        df[c] = df[c].fillna(df.groupby("Patient_ID")[c].transform("mean"))

    df["ShockIndex"] = df["HR"] / df["SBP"].replace(0, 1)
    df["ModShockIndex"] = df["HR"] / df["MAP"].replace(0, 1)
    df["PulsePressure"] = df["SBP"] - df["DBP"]

    new_features = ["HR", "MAP", "Resp", "ShockIndex", "ModShockIndex"]

    for c in new_features:
        df[f"{c}_diff6"] = df.groupby("Patient_ID")[c].diff(6).fillna(0)

    return df

ALL_FEATURES = FEATURES + [
    "HR_diff6", "MAP_diff6", "Resp_diff6",
    "ShockIndex", "ModShockIndex", "ShockIndex_diff6", "ModShockIndex_diff6"
]

# =========================
# STEP 3: WINDOWING
# =========================
def make_windows(df, balance=False, max_per_patient=16, min_signal=0.15):
    X_pos, X_neg = [], []

    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < WINDOW:
            continue

        n = min((len(g)-WINDOW)//STRIDE + 1, max_per_patient)
        collected = 0

        for i in range(0, len(g)-WINDOW+1, STRIDE):
            if collected >= n:
                break

            x = g.iloc[i:i+WINDOW][ALL_FEATURES].values.astype(np.float32)
            y = g.iloc[i:i+WINDOW]["SepsisLabel_shifted"].max()

            if balance:
                signal = np.abs(x[-1] - x[-7]).mean()
                if signal < min_signal:
                    continue

            (X_pos if y == 1 else X_neg).append(x)
            collected += 1

    if balance:
        n = min(len(X_pos), len(X_neg))
        X_pos, X_neg = X_pos[:n], X_neg[:n]

    X = np.array(X_pos + X_neg)
    y = np.array([1]*len(X_pos) + [0]*len(X_neg))
    return X, y

# =========================
# STEP 4: CLASSICAL TRANSFORMER MODEL
# =========================
class ClassicalTransformer(nn.Module):
    def __init__(self, n_features, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=DROPOUT):
        super().__init__()
        
        # 1. Project raw features to the Transformer's expected dimension
        self.input_projection = nn.Linear(n_features, d_model)
        
        # 2. Positional Encoding (Learnable parameter since window is fixed at 12)
        self.pos_encoder = nn.Parameter(torch.zeros(1, WINDOW, d_model))
        
        # 3. Transformer Encoder Blocks
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout, 
            batch_first=True # Crucial: ensures inputs are (Batch, Seq, Features)
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        # 4. Final Classification Head
        self.cls = nn.Linear(d_model, 1)

    def forward(self, x):
        # x shape: (Batch, Window=12, Features=22)
        
        # Map features to d_model space
        x = self.input_projection(x) 
        
        # Add positional context so the model knows the order of the 12 hours
        x = x + self.pos_encoder 
        
        # Pass through Self-Attention layers
        x = self.transformer_encoder(x) 
        
        # Extract the representation of the final time step (t=-1) for prediction
        last_out = x[:, -1, :] 
        
        logits = self.cls(last_out)
        return logits.squeeze(1)

# =========================
# LOSS
# =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.gamma = gamma

    def forward(self, logits, y):
        bce = self.bce(logits, y)
        pt = torch.exp(-bce)
        return ((1-pt)**self.gamma * bce).mean()

# =========================
# 5-FOLD PATIENT CV + ENSEMBLE
# =========================
from sklearn.model_selection import GroupShuffleSplit
import copy

process = psutil.Process(os.getpid())

# 1. Prepare Data
df = preprocess(apply_clinical_labeling(df))

# 2. Create Hold-Out Test Set (20%)
print("✂️ Splitting Hold-Out Test Set (20%)...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df.Patient_ID))

df_dev = df.iloc[train_idx]
df_test = df.iloc[test_idx]

print(f"Dataset Shape: Dev {df_dev.shape}, Test {df_test.shape}")

gkf = GroupKFold(n_splits=5)
saved_checkpoints = []

# --- TRAINING TIME / CONVERGENCE METRICS ---
global_train_start = time.time()
converged_fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_dev, groups=df_dev.Patient_ID)):
    print(f"\n🚀 Training Ensemble Model {fold+1}/5...")

    df_tr = df_dev.iloc[tr_idx]

    Xtr, ytr = make_windows(df_tr, balance=True, max_per_patient=16)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr.reshape(-1, Xtr.shape[-1])).reshape(Xtr.shape)

    model = ClassicalTransformer(Xtr.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))
    focal = FocalLoss()

    def loss_fn(logits, y):
        return ALPHA*bce(logits, y) + (1-ALPHA)*focal(logits, y)

    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).float()),
        batch_size=BATCH_SIZE, shuffle=True
    )

    converged_epoch = None
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        model.train()
        all_logits = []
        all_targets = []

        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            all_logits.append(logits.detach().cpu())
            all_targets.append(yb.detach().cpu())

        epoch_time = time.time() - epoch_start
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Fold {fold+1}, Epoch {epoch+1}/{EPOCHS}, epoch time: {epoch_time:.3f} s, loss: {loss.item():.4f}")

        with torch.no_grad():
            probs_epoch = torch.sigmoid(torch.cat(all_logits)).numpy()
            y_epoch = torch.cat(all_targets).numpy()
            try:
                train_auroc_epoch = roc_auc_score(y_epoch, probs_epoch)
            except ValueError:
                train_auroc_epoch = np.nan

        if converged_epoch is None and not np.isnan(train_auroc_epoch) and train_auroc_epoch >= TARGET_AUROC:
            converged_epoch = epoch + 1  
            
    converged_fold_epochs.append(converged_epoch)

    checkpoint = {
        'model_state': copy.deepcopy(model.state_dict()),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }
    saved_checkpoints.append(checkpoint)
    print(f"✅ Model {fold+1} Saved. Converged epoch (train AUROC ≥ {TARGET_AUROC}): {converged_epoch}")

global_train_end = time.time()
total_training_time = global_train_end - global_train_start
print(f"\n⏱ Total training time (5 folds): {total_training_time/60:.2f} minutes")

# =========================
# ENSEMBLE VOTING (INFERENCE)
# =========================
print(f"\n🔮 Running Ensemble Voting on {df_test.Patient_ID.nunique()} Test Patients...")

X_test_raw, y_test = make_windows(df_test, balance=False, max_per_patient=999)

ensemble_probs = np.zeros(len(y_test))

# --- Inference time & resource tracking ---
mem_before = process.memory_info().rss
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

inf_start = time.time()

for i, checkpoint in enumerate(saved_checkpoints):
    scaler = StandardScaler()
    scaler.mean_ = checkpoint['scaler_mean']
    scaler.scale_ = checkpoint['scaler_scale']

    X_test_scaled = scaler.transform(
        X_test_raw.reshape(-1, X_test_raw.shape[-1])
    ).reshape(X_test_raw.shape)

    model = ClassicalTransformer(X_test_scaled.shape[-1]).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    with torch.no_grad():
        logits = model(torch.tensor(X_test_scaled).to(DEVICE))
        logits = torch.nan_to_num(logits, nan=0.0, posinf=10.0, neginf=-10.0)
        probs = torch.sigmoid(logits).cpu().numpy()

    ensemble_probs += probs

if torch.cuda.is_available():
    torch.cuda.synchronize()
inf_end = time.time()

ensemble_probs /= len(saved_checkpoints)

mem_after = process.memory_info().rss
cpu_mem_used = (mem_after - mem_before) / (1024**2)  # MB

if torch.cuda.is_available():
    gpu_peak_mem = torch.cuda.max_memory_allocated() / (1024**2)
else:
    gpu_peak_mem = None

inference_time = inf_end - inf_start
latency_per_sample = inference_time / len(y_test)

# Model size (parameters only, single model)
model_for_size = ClassicalTransformer(X_test_raw.shape[-1]).to(DEVICE)
total_params = sum(p.numel() for p in model_for_size.parameters())
model_size_mb = total_params * 4 / (1024**2)

print("\n⏱ Time Metrics")
print(f"Inference time (ensemble, test set): {inference_time:.4f} s")
print(f"Latency per sample: {latency_per_sample*1e3:.3f} ms")
print(f"Total training time (all folds): {total_training_time:.2f} s")
print(f"Converged epochs per fold: {converged_fold_epochs}")

print("\n💾 Resource Metrics")
print(f"CPU memory used during inference: {cpu_mem_used:.2f} MB")
if gpu_peak_mem is not None:
    print(f"GPU peak memory during inference: {gpu_peak_mem:.2f} MB")
print(f"Model size (single model parameters): {model_size_mb:.2f} MB")
print(f"Total parameters: {total_params}")

# =========================
# FINAL METRICS
# =========================
auroc = roc_auc_score(y_test, ensemble_probs)
auprc = average_precision_score(y_test, ensemble_probs)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_probs > 0.5, average="binary", zero_division=0
)

print("\n" + "="*40)
print(f"🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟")
print("="*40)
print(f"Test AUROC: {auroc:.3f}")
print(f"Test AUPRC: {auprc:.3f}")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
print("="*40)

✂️ Splitting Hold-Out Test Set (20%)...
Dataset Shape: Dev (1241213, 52), Test (310997, 52)

🚀 Training Ensemble Model 1/5...
Fold 1, Epoch 1/120, epoch time: 1.491 s, loss: 0.2540
Fold 1, Epoch 10/120, epoch time: 1.447 s, loss: 0.0804
Fold 1, Epoch 20/120, epoch time: 1.443 s, loss: 0.1662
Fold 1, Epoch 30/120, epoch time: 1.386 s, loss: 0.0571
Fold 1, Epoch 40/120, epoch time: 1.415 s, loss: 0.0147
Fold 1, Epoch 50/120, epoch time: 1.616 s, loss: 0.0284
Fold 1, Epoch 60/120, epoch time: 1.605 s, loss: 0.0023
Fold 1, Epoch 70/120, epoch time: 1.406 s, loss: 0.1444
Fold 1, Epoch 80/120, epoch time: 1.593 s, loss: 0.0125
Fold 1, Epoch 90/120, epoch time: 2.049 s, loss: 0.0273
Fold 1, Epoch 100/120, epoch time: 1.555 s, loss: 0.0012
Fold 1, Epoch 110/120, epoch time: 1.443 s, loss: 0.0079
Fold 1, Epoch 120/120, epoch time: 1.723 s, loss: 0.0018
✅ Model 1 Saved. Converged epoch (train AUROC ≥ 0.85): 4

🚀 Training Ensemble Model 2/5...
Fold 2, Epoch 1/120, epoch time: 2.126 s, loss: 0.259

XGBOOST

In [5]:
import numpy as np
import pandas as pd
import gc, warnings, time, os, psutil
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support
)

import xgboost as xgb
import torch # Kept strictly for GPU memory tracking parity with previous scripts

# =========================
# GLOBAL SETTINGS
# =========================
SEED = 42
np.random.seed(SEED)

WINDOW = 12 
STRIDE = 6
POS_WEIGHT = 2.0

# XGBoost specific settings
N_ESTIMATORS = 150
MAX_DEPTH = 5
LEARNING_RATE = 0.1

# =========================
# STEP 1: CLINICAL LABELING
# =========================
def apply_clinical_labeling(df):
    df = df.copy()
    df["SepsisLabel_shifted"] = 0
    df["use_for_training"] = 1

    for pid, g in df.groupby("Patient_ID"):
        g = g.reset_index()
        onset_idx = g[g["SepsisLabel"] == 1].index

        for t in onset_idx:
            pos_start = max(0, t - 3)
            df.loc[g.loc[pos_start:t, "index"], "SepsisLabel_shifted"] = 1

            amb_start = max(0, t - 6)
            df.loc[g.loc[amb_start:pos_start-1, "index"], "use_for_training"] = 0

    return df

# =========================
# STEP 2: PREPROCESSING
# =========================
FEATURES = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
    'pH','PaCO2','BUN','Creatinine','Lactate',
    'WBC','Platelets','Age'
]

def preprocess(df):
    df = df.copy()

    for c in FEATURES:
        df[c] = df.groupby("Patient_ID")[c].ffill().bfill()
        df[c] = df[c].fillna(df.groupby("Patient_ID")[c].transform("mean"))

    df["ShockIndex"] = df["HR"] / df["SBP"].replace(0, 1)
    df["ModShockIndex"] = df["HR"] / df["MAP"].replace(0, 1)
    df["PulsePressure"] = df["SBP"] - df["DBP"]

    new_features = ["HR", "MAP", "Resp", "ShockIndex", "ModShockIndex"]

    for c in new_features:
        df[f"{c}_diff6"] = df.groupby("Patient_ID")[c].diff(6).fillna(0)

    return df

ALL_FEATURES = FEATURES + [
    "HR_diff6", "MAP_diff6", "Resp_diff6",
    "ShockIndex", "ModShockIndex", "ShockIndex_diff6", "ModShockIndex_diff6"
]

# =========================
# STEP 3: WINDOWING
# =========================
def make_windows(df, balance=False, max_per_patient=16, min_signal=0.15):
    X_pos, X_neg = [], []

    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < WINDOW:
            continue

        n = min((len(g)-WINDOW)//STRIDE + 1, max_per_patient)
        collected = 0

        for i in range(0, len(g)-WINDOW+1, STRIDE):
            if collected >= n:
                break

            x = g.iloc[i:i+WINDOW][ALL_FEATURES].values.astype(np.float32)
            y = g.iloc[i:i+WINDOW]["SepsisLabel_shifted"].max()

            if balance:
                signal = np.abs(x[-1] - x[-7]).mean()
                if signal < min_signal:
                    continue

            (X_pos if y == 1 else X_neg).append(x)
            collected += 1

    if balance:
        n = min(len(X_pos), len(X_neg))
        X_pos, X_neg = X_pos[:n], X_neg[:n]

    X = np.array(X_pos + X_neg)
    y = np.array([1]*len(X_pos) + [0]*len(X_neg))
    return X, y

# =========================
# 5-FOLD PATIENT CV + ENSEMBLE
# =========================
from sklearn.model_selection import GroupShuffleSplit
import copy

process = psutil.Process(os.getpid())

# 1. Prepare Data
df = preprocess(apply_clinical_labeling(df))

# 2. Create Hold-Out Test Set (20%)
print("✂️ Splitting Hold-Out Test Set (20%)...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df.Patient_ID))

df_dev = df.iloc[train_idx]
df_test = df.iloc[test_idx]

print(f"Dataset Shape: Dev {df_dev.shape}, Test {df_test.shape}")

gkf = GroupKFold(n_splits=5)
saved_checkpoints = []

# --- TRAINING TIME METRICS ---
global_train_start = time.time()

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_dev, groups=df_dev.Patient_ID)):
    print(f"\n🚀 Training Ensemble Model {fold+1}/5...")
    fold_start = time.time()

    df_tr = df_dev.iloc[tr_idx]

    Xtr, ytr = make_windows(df_tr, balance=True, max_per_patient=16)

    # Flatten the 3D window (Batch, 12, 22) into 2D (Batch, 264) for XGBoost
    Xtr_flat = Xtr.reshape(Xtr.shape[0], -1)

    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(Xtr_flat)

    # Initialize XGBoost Classifier
    model = xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE,
        scale_pos_weight=POS_WEIGHT,
        eval_metric="aucpr",
        random_state=SEED,
        n_jobs=-1, # Use all CPU cores
        tree_method="hist" # Fast histogram optimized algorithm
    )

    # Train (No epochs loop needed for gradient boosting)
    model.fit(Xtr_scaled, ytr)
    
    fold_time = time.time() - fold_start
    print(f"✅ Model {fold+1} Trained in {fold_time:.2f} seconds.")

    checkpoint = {
        'model': copy.deepcopy(model),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }
    saved_checkpoints.append(checkpoint)

global_train_end = time.time()
total_training_time = global_train_end - global_train_start
print(f"\n⏱ Total training time (5 folds): {total_training_time/60:.2f} minutes")

# =========================
# ENSEMBLE VOTING (INFERENCE)
# =========================
print(f"\n🔮 Running Ensemble Voting on {df_test.Patient_ID.nunique()} Test Patients...")

X_test_raw, y_test = make_windows(df_test, balance=False, max_per_patient=999)

# Flatten test data
X_test_flat = X_test_raw.reshape(X_test_raw.shape[0], -1)

ensemble_probs = np.zeros(len(y_test))

# --- Inference time & resource tracking ---
mem_before = process.memory_info().rss
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

inf_start = time.time()

for i, checkpoint in enumerate(saved_checkpoints):
    scaler = StandardScaler()
    scaler.mean_ = checkpoint['scaler_mean']
    scaler.scale_ = checkpoint['scaler_scale']

    X_test_scaled = scaler.transform(X_test_flat)

    model = checkpoint['model']

    # Predict probabilities (extracting the positive class probability)
    probs = model.predict_proba(X_test_scaled)[:, 1]

    ensemble_probs += probs

if torch.cuda.is_available():
    torch.cuda.synchronize()
inf_end = time.time()

ensemble_probs /= len(saved_checkpoints)

mem_after = process.memory_info().rss
cpu_mem_used = (mem_after - mem_before) / (1024**2)  # MB

if torch.cuda.is_available():
    gpu_peak_mem = torch.cuda.max_memory_allocated() / (1024**2)
else:
    gpu_peak_mem = None

inference_time = inf_end - inf_start
latency_per_sample = inference_time / len(y_test)

# Calculate Model Size in MB (by saving a dummy file)
temp_file = "temp_xgb_model.json"
saved_checkpoints[0]['model'].save_model(temp_file)
model_size_mb = os.path.getsize(temp_file) / (1024**2)
os.remove(temp_file)

print("\n⏱ Time Metrics")
print(f"Inference time (ensemble, test set): {inference_time:.4f} s")
print(f"Latency per sample: {latency_per_sample*1e3:.3f} ms")
print(f"Total training time (all folds): {total_training_time:.2f} s")

print("\n💾 Resource Metrics")
print(f"CPU memory used during inference: {cpu_mem_used:.2f} MB")
if gpu_peak_mem is not None:
    print(f"GPU peak memory during inference: {gpu_peak_mem:.2f} MB")
print(f"Model size (single model on disk): {model_size_mb:.2f} MB")
print(f"Total Trees per model: {N_ESTIMATORS}")

# =========================
# FINAL METRICS
# =========================
auroc = roc_auc_score(y_test, ensemble_probs)
auprc = average_precision_score(y_test, ensemble_probs)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_probs > 0.5, average="binary", zero_division=0
)

print("\n" + "="*40)
print(f"🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟")
print("="*40)
print(f"Test AUROC: {auroc:.3f}")
print(f"Test AUPRC: {auprc:.3f}")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
print("="*40)

✂️ Splitting Hold-Out Test Set (20%)...
Dataset Shape: Dev (1241213, 52), Test (310997, 52)

🚀 Training Ensemble Model 1/5...
✅ Model 1 Trained in 51.88 seconds.

🚀 Training Ensemble Model 2/5...
✅ Model 2 Trained in 46.63 seconds.

🚀 Training Ensemble Model 3/5...
✅ Model 3 Trained in 45.21 seconds.

🚀 Training Ensemble Model 4/5...
✅ Model 4 Trained in 45.38 seconds.

🚀 Training Ensemble Model 5/5...
✅ Model 5 Trained in 45.32 seconds.

⏱ Total training time (5 folds): 3.92 minutes

🔮 Running Ensemble Voting on 8068 Test Patients...

⏱ Time Metrics
Inference time (ensemble, test set): 0.5647 s
Latency per sample: 0.014 ms
Total training time (all folds): 235.33 s

💾 Resource Metrics
CPU memory used during inference: 40.73 MB
GPU peak memory during inference: 0.00 MB
Model size (single model on disk): 0.40 MB
Total Trees per model: 150

🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟
Test AUROC: 0.737
Test AUPRC: 0.019
Precision: 0.009, Recall: 0.897, F1: 0.017


MLP

In [6]:
import numpy as np
import pandas as pd
import gc, warnings, time, os, psutil
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# =========================
# GLOBAL SETTINGS
# =========================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
WINDOW = 12 
STRIDE = 6
POS_WEIGHT = 2.0
ALPHA = 0.2
EPOCHS = 120
BATCH_SIZE = 8

TARGET_AUROC = 0.85  # for convergence time tracking

# =========================
# STEP 1: CLINICAL LABELING
# =========================
def apply_clinical_labeling(df):
    df = df.copy()
    df["SepsisLabel_shifted"] = 0
    df["use_for_training"] = 1

    for pid, g in df.groupby("Patient_ID"):
        g = g.reset_index()
        onset_idx = g[g["SepsisLabel"] == 1].index

        for t in onset_idx:
            pos_start = max(0, t - 3)
            df.loc[g.loc[pos_start:t, "index"], "SepsisLabel_shifted"] = 1

            amb_start = max(0, t - 6)
            df.loc[g.loc[amb_start:pos_start-1, "index"], "use_for_training"] = 0

    return df

# =========================
# STEP 2: PREPROCESSING
# =========================
FEATURES = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp',
    'pH','PaCO2','BUN','Creatinine','Lactate',
    'WBC','Platelets','Age'
]

def preprocess(df):
    df = df.copy()

    for c in FEATURES:
        df[c] = df.groupby("Patient_ID")[c].ffill().bfill()
        df[c] = df[c].fillna(df.groupby("Patient_ID")[c].transform("mean"))

    df["ShockIndex"] = df["HR"] / df["SBP"].replace(0, 1)
    df["ModShockIndex"] = df["HR"] / df["MAP"].replace(0, 1)
    df["PulsePressure"] = df["SBP"] - df["DBP"]

    new_features = ["HR", "MAP", "Resp", "ShockIndex", "ModShockIndex"]

    for c in new_features:
        df[f"{c}_diff6"] = df.groupby("Patient_ID")[c].diff(6).fillna(0)

    return df

ALL_FEATURES = FEATURES + [
    "HR_diff6", "MAP_diff6", "Resp_diff6",
    "ShockIndex", "ModShockIndex", "ShockIndex_diff6", "ModShockIndex_diff6"
]

# =========================
# STEP 3: WINDOWING
# =========================
def make_windows(df, balance=False, max_per_patient=16, min_signal=0.15):
    X_pos, X_neg = [], []

    for pid, g in df.groupby("Patient_ID"):
        g = g[g["use_for_training"] == 1].reset_index(drop=True)
        if len(g) < WINDOW:
            continue

        n = min((len(g)-WINDOW)//STRIDE + 1, max_per_patient)
        collected = 0

        for i in range(0, len(g)-WINDOW+1, STRIDE):
            if collected >= n:
                break

            x = g.iloc[i:i+WINDOW][ALL_FEATURES].values.astype(np.float32)
            y = g.iloc[i:i+WINDOW]["SepsisLabel_shifted"].max()

            if balance:
                signal = np.abs(x[-1] - x[-7]).mean()
                if signal < min_signal:
                    continue

            (X_pos if y == 1 else X_neg).append(x)
            collected += 1

    if balance:
        n = min(len(X_pos), len(X_neg))
        X_pos, X_neg = X_pos[:n], X_neg[:n]

    X = np.array(X_pos + X_neg)
    y = np.array([1]*len(X_pos) + [0]*len(X_neg))
    return X, y

# =========================
# STEP 4: CLASSICAL MICRO-MLP MODEL (~200 Params)
# =========================
class ClassicalMicroMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        
        # 1. Spatio-Temporal Adapter: exactly like the Quantum model
        # Input: 2 * 22 features = 44. Output: 4
        # Parameters: (44 * 4 weights) + 4 biases = 180 parameters
        self.adapter = nn.Linear(2 * n_features, 4)
        self.relu1 = nn.ReLU()
        
        # 2. Small Hidden Layer: replacing the 4-qubit Quantum Circuit
        # Input: 4. Output: 3.
        # Parameters: (4 * 3 weights) + 3 biases = 15 parameters
        self.hidden = nn.Linear(4, 3) 
        self.relu2 = nn.ReLU()
        
        # 3. Classification Head
        # Input: 3. Output: 1.
        # Parameters: (3 * 1 weight) + 1 bias = 4 parameters
        self.cls = nn.Linear(3, 1)
        
        # TOTAL PARAMETERS: 180 + 15 + 4 = 199.
        # (Directly comparable to the Quantum model's 197 parameters)

    def forward(self, x):
        # Extract Current State and Past State (exactly like the Quantum script)
        x_now = x[:, -1, :]
        x_prev = x[:, -7, :]

        # Classical Spatio-Temporal Fusion
        z = torch.cat([x_now, x_now - x_prev], dim=1)

        # Pass through Micro-MLP layers
        z = self.adapter(z)
        z = self.relu1(z)
        
        z = self.hidden(z)
        z = self.relu2(z)

        logits = self.cls(z)
        return logits.squeeze(1)

# =========================
# LOSS
# =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.gamma = gamma

    def forward(self, logits, y):
        bce = self.bce(logits, y)
        pt = torch.exp(-bce)
        return ((1-pt)**self.gamma * bce).mean()

# =========================
# 5-FOLD PATIENT CV + ENSEMBLE
# =========================
from sklearn.model_selection import GroupShuffleSplit
import copy

process = psutil.Process(os.getpid())

# 1. Prepare Data
df = preprocess(apply_clinical_labeling(df))

# 2. Create Hold-Out Test Set (20%)
print("✂️ Splitting Hold-Out Test Set (20%)...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, groups=df.Patient_ID))

df_dev = df.iloc[train_idx]
df_test = df.iloc[test_idx]

print(f"Dataset Shape: Dev {df_dev.shape}, Test {df_test.shape}")

gkf = GroupKFold(n_splits=5)
saved_checkpoints = []

# --- TRAINING TIME / CONVERGENCE METRICS ---
global_train_start = time.time()
converged_fold_epochs = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_dev, groups=df_dev.Patient_ID)):
    print(f"\n🚀 Training Ensemble Model {fold+1}/5...")

    df_tr = df_dev.iloc[tr_idx]

    Xtr, ytr = make_windows(df_tr, balance=True, max_per_patient=16)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr.reshape(-1, Xtr.shape[-1])).reshape(Xtr.shape)

    model = ClassicalMicroMLP(Xtr.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT]).to(DEVICE))
    focal = FocalLoss()

    def loss_fn(logits, y):
        return ALPHA*bce(logits, y) + (1-ALPHA)*focal(logits, y)

    loader = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).float()),
        batch_size=BATCH_SIZE, shuffle=True
    )

    converged_epoch = None
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        model.train()
        all_logits = []
        all_targets = []

        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

            all_logits.append(logits.detach().cpu())
            all_targets.append(yb.detach().cpu())

        epoch_time = time.time() - epoch_start
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Fold {fold+1}, Epoch {epoch+1}/{EPOCHS}, epoch time: {epoch_time:.3f} s, loss: {loss.item():.4f}")

        with torch.no_grad():
            probs_epoch = torch.sigmoid(torch.cat(all_logits)).numpy()
            y_epoch = torch.cat(all_targets).numpy()
            try:
                train_auroc_epoch = roc_auc_score(y_epoch, probs_epoch)
            except ValueError:
                train_auroc_epoch = np.nan

        if converged_epoch is None and not np.isnan(train_auroc_epoch) and train_auroc_epoch >= TARGET_AUROC:
            converged_epoch = epoch + 1  
            
    converged_fold_epochs.append(converged_epoch)

    checkpoint = {
        'model_state': copy.deepcopy(model.state_dict()),
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_
    }
    saved_checkpoints.append(checkpoint)
    print(f"✅ Model {fold+1} Saved. Converged epoch (train AUROC ≥ {TARGET_AUROC}): {converged_epoch}")

global_train_end = time.time()
total_training_time = global_train_end - global_train_start
print(f"\n⏱ Total training time (5 folds): {total_training_time/60:.2f} minutes")

# =========================
# ENSEMBLE VOTING (INFERENCE)
# =========================
print(f"\n🔮 Running Ensemble Voting on {df_test.Patient_ID.nunique()} Test Patients...")

X_test_raw, y_test = make_windows(df_test, balance=False, max_per_patient=999)

ensemble_probs = np.zeros(len(y_test))

# --- Inference time & resource tracking ---
mem_before = process.memory_info().rss
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

inf_start = time.time()

for i, checkpoint in enumerate(saved_checkpoints):
    scaler = StandardScaler()
    scaler.mean_ = checkpoint['scaler_mean']
    scaler.scale_ = checkpoint['scaler_scale']

    X_test_scaled = scaler.transform(
        X_test_raw.reshape(-1, X_test_raw.shape[-1])
    ).reshape(X_test_raw.shape)

    model = ClassicalMicroMLP(X_test_scaled.shape[-1]).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    with torch.no_grad():
        logits = model(torch.tensor(X_test_scaled).to(DEVICE))
        logits = torch.nan_to_num(logits, nan=0.0, posinf=10.0, neginf=-10.0)
        probs = torch.sigmoid(logits).cpu().numpy()

    ensemble_probs += probs

if torch.cuda.is_available():
    torch.cuda.synchronize()
inf_end = time.time()

ensemble_probs /= len(saved_checkpoints)

mem_after = process.memory_info().rss
cpu_mem_used = (mem_after - mem_before) / (1024**2)  # MB

if torch.cuda.is_available():
    gpu_peak_mem = torch.cuda.max_memory_allocated() / (1024**2)
else:
    gpu_peak_mem = None

inference_time = inf_end - inf_start
latency_per_sample = inference_time / len(y_test)

# Model size (parameters only, single model)
model_for_size = ClassicalMicroMLP(X_test_raw.shape[-1]).to(DEVICE)
total_params = sum(p.numel() for p in model_for_size.parameters())
model_size_mb = total_params * 4 / (1024**2)

print("\n⏱ Time Metrics")
print(f"Inference time (ensemble, test set): {inference_time:.4f} s")
print(f"Latency per sample: {latency_per_sample*1e3:.3f} ms")
print(f"Total training time (all folds): {total_training_time:.2f} s")
print(f"Converged epochs per fold: {converged_fold_epochs}")

print("\n💾 Resource Metrics")
print(f"CPU memory used during inference: {cpu_mem_used:.2f} MB")
if gpu_peak_mem is not None:
    print(f"GPU peak memory during inference: {gpu_peak_mem:.2f} MB")
print(f"Model size (single model parameters): {model_size_mb:.2f} MB")
print(f"Total parameters: {total_params}")

# =========================
# FINAL METRICS
# =========================
auroc = roc_auc_score(y_test, ensemble_probs)
auprc = average_precision_score(y_test, ensemble_probs)

prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, ensemble_probs > 0.5, average="binary", zero_division=0
)

print("\n" + "="*40)
print(f"🌟 FINAL ENSEMBLE RESULTS (n=5 Models) 🌟")
print("="*40)
print(f"Test AUROC: {auroc:.3f}")
print(f"Test AUPRC: {auprc:.3f}")
print(f"Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")
print("="*40)

✂️ Splitting Hold-Out Test Set (20%)...
Dataset Shape: Dev (1241213, 52), Test (310997, 52)

🚀 Training Ensemble Model 1/5...
Fold 1, Epoch 1/120, epoch time: 0.298 s, loss: 0.3263
Fold 1, Epoch 10/120, epoch time: 0.202 s, loss: 0.1644
Fold 1, Epoch 20/120, epoch time: 0.190 s, loss: 0.1335
Fold 1, Epoch 30/120, epoch time: 0.183 s, loss: 0.2216
Fold 1, Epoch 40/120, epoch time: 0.218 s, loss: 0.1904
Fold 1, Epoch 50/120, epoch time: 0.188 s, loss: 0.0211
Fold 1, Epoch 60/120, epoch time: 0.222 s, loss: 0.0824
Fold 1, Epoch 70/120, epoch time: 0.192 s, loss: 0.0205
Fold 1, Epoch 80/120, epoch time: 0.226 s, loss: 0.0135
Fold 1, Epoch 90/120, epoch time: 0.194 s, loss: 0.0166
Fold 1, Epoch 100/120, epoch time: 0.192 s, loss: 0.2080
Fold 1, Epoch 110/120, epoch time: 0.182 s, loss: 0.0950
Fold 1, Epoch 120/120, epoch time: 0.206 s, loss: 0.0640
✅ Model 1 Saved. Converged epoch (train AUROC ≥ 0.85): 12

🚀 Training Ensemble Model 2/5...
Fold 2, Epoch 1/120, epoch time: 0.202 s, loss: 0.39